# Notebook 4: Adversarial Defense

Evaluate defense mechanisms against adversarial attacks.

## Key Finding: Simple defenses (adversarial training + feature smoothing) are insufficient.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt

from src.dataset.loader import get_dataset
from src.dataset.preprocess import normalize_features, create_train_val_test_split
from src.models.detector import GCNDetector
from src.models.attack import FeatureAttack
from src.models.defense import AdversarialTraining, RobustDetector
from src.utils.metrics import compute_attack_success_rate, compute_metrics
from src.utils.visualization import plot_attack_results

In [ ]:
# Setup
device = "cuda:0" if torch.cuda.is_available() else "cpu"
data, labels, desc = get_dataset(seed=42)
data = normalize_features(data).to(device)
target_mask = labels > 0
train_mask, val_mask, test_mask = create_train_val_test_split(data)
train_mask, val_mask, test_mask = train_mask.to(device), val_mask.to(device), test_mask.to(device)
print(desc)

In [ ]:
# Evaluate baseline under feature attack
checkpoint = torch.load("baseline_model.pth", map_location="cpu")
baseline_model = GCNDetector(in_channels=16, hidden_channels=64, out_channels=3)
baseline_model.load_state_dict(checkpoint["model_state"])
baseline_model = baseline_model.to(device)
baseline_model.eval()

print("=== Baseline Under Feature Attack ===")
for bound in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    feat_attack = FeatureAttack(baseline_model, perturbation_bound=bound, n_steps=20)
    perturbed, info = feat_attack.attack(data, target_mask, device)
    asr, _ = compute_attack_success_rate(baseline_model, data, target_mask, perturbed, device)
    print(f"  Bound={bound:.1f}: ASR={asr:.4f}")

In [ ]:
# Train defended model
import torch.nn as nn
import torch.optim as optim

base_model = GCNDetector(in_channels=16, hidden_channels=64, out_channels=3)
defended_model = RobustDetector(base_model, smoothing_alpha=0.3, smoothing_steps=5)
defended_model = defended_model.to(device)

class_counts = torch.bincount(data.y)
class_weights = 1.0 / (class_counts.float() + 1e-8)
class_weights = class_weights / class_weights.sum() * len(class_weights)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.Adam(defended_model.parameters(), lr=0.01, weight_decay=5e-4)

adv_trainer = AdversarialTraining(defended_model, attack_budget=0.03, feature_bound=2.0, attack_freq=2, alpha=1.0)

best_val_f1 = 0.0
best_state = None
for epoch in range(200):
    defended_model.train()
    total_loss, clean_loss = adv_trainer.train_step(data, train_mask, optimizer, epoch, device)
    defended_model.eval()
    with torch.no_grad():
        val_logits = defended_model(data.x, data.edge_index)
        val_preds = val_logits.argmax(dim=1)
        val_metrics = compute_metrics(data.y[val_mask], val_preds[val_mask])
    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_state = {k: v.clone() for k, v in defended_model.state_dict().items()}

if best_state:
    defended_model.load_state_dict(best_state)
print(f"Defended model best val F1: {best_val_f1:.4f}")

In [ ]:
# Evaluate defended model under attack
print("\n=== Defended Model Under Feature Attack ===")
defended_model.eval()
for bound in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    feat_attack = FeatureAttack(defended_model, perturbation_bound=bound, n_steps=20)
    perturbed, info = feat_attack.attack(data, target_mask, device)
    asr, _ = compute_attack_success_rate(defended_model, data, target_mask, perturbed, device)
    print(f"  Bound={bound:.1f}: ASR={asr:.4f}")